# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logging
import matplotlib.font_manager
import seaborn as sns
from pathlib import Path
from scipy.stats import wasserstein_distance_nd

from typing import Callable, List, Dict, Union
import warnings


#Rdkit + clustering

from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import Draw
from IPython.display import display
from rdkit.ML.Cluster import Butina
from scipy.spatial.distance import pdist, squareform
from rdkit.DataStructs import FingerprintSimilarity
from sklearn.cluster import DBSCAN,HDBSCAN
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.manifold import TSNE
from sklearn.decomposition import LatentDirichletAllocation
from sklearn_extra.cluster import KMedoids
# import kmedoids
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from sklearn.metrics import jaccard_score
from scipy.spatial.distance import euclidean
from sklearn.mixture import GaussianMixture

#Plotly
import base64
import textwrap
from io import BytesIO
from typing import Callable
import itertools
import re

import pandas as pd
import numpy as np
from pandas.core.groupby import DataFrameGroupBy
from plotly.graph_objects import Figure
import plotly.graph_objects as go
import plotly.express as px

from rdkit import Chem
from rdkit.Chem.rdChemReactions import ReactionFromSmarts
from rdkit.Chem.rdchem import Mol
from rdkit.Chem.Draw import rdMolDraw2D
import random

##scaler
from sklearn.preprocessing import StandardScaler


HERE = Path(__file__).parent.resolve()

from code_python.visualization.visualization_setting import set_plot_style



ModuleNotFoundError: No module named 'code_python'

# Tools and Functions

In [4]:
def weighted_jaccard(u, v,eps: float = 1e-6):
    min_sum = np.sum(np.minimum(u, v))
    max_sum = np.sum(np.maximum(u, v))
    return 1 - ((min_sum+eps) / (max_sum+eps))


def _get_scores(fp_vector:np.ndarray, predicted_clusters, metric: Union[str, Callable]):
    si_score = silhouette_score(fp_vector, predicted_clusters, metric=metric)
    db_score = davies_bouldin_score(fp_vector, predicted_clusters)
    ch_score = calinski_harabasz_score(fp_vector, predicted_clusters)
    return si_score, db_score, ch_score
    

naming: dict = {
    'jaccard' : 'jaccard',
    'euclidean': 'euclidean',
    'weighted_jaccard': weighted_jaccard,
    'KMedoids': KMedoids,
    'DBSCAN': DBSCAN,
    'HDBSCAN': HDBSCAN,
    'GaussianMixture':GaussianMixture,

}

**Plot simple-tsne**

In [5]:
x_y_label_format = {'fontsize':20,'fontweight':'bold'}
title_label_format = {'fontsize':22,'fontweight':'bold'}
x_y_tick_format = {'labelsize':18}

In [6]:
def plot_clusters(fp_vector,
                  predicted_clusters,
                  unique_clusters,
                  method: str,
                  target: str,
                  vector: str,
                  metric: str,
                  medoid_indices=None,
                  perplexity: int = 20,
                  loc ='upper center'
                  ):
    """
    Perform t-SNE computation and plot clusters. Mark medoid points with a red star.
    """
    tsne = TSNE(n_components=2, metric=naming[metric], random_state=42, perplexity=perplexity)
    tsne_results = tsne.fit_transform(fp_vector)

    plot_title = f"{vector} cluster with {method}{len(unique_clusters)}"
    unique_clusters, label_counts = np.unique(predicted_clusters, return_counts=True)
    palette = sns.color_palette("Set2", len(unique_clusters))
    cluster_colors = {label: palette[i] for i, label in enumerate(unique_clusters)}

    fig, ax = plt.subplots(figsize=(10, 8))

    for label in unique_clusters:
        indices = predicted_clusters == label
        ax.scatter(tsne_results[indices, 0], tsne_results[indices, 1],
                   color=cluster_colors[label], label=f'Cluster {label}', alpha=0.7)

    # Mark medoid points with a red star
    if medoid_indices is not None:
        ax.scatter(tsne_results[medoid_indices, 0], tsne_results[medoid_indices, 1],
                   color='red', marker='*', s=200, label='Medoids')


    ncol = len(unique_clusters) + (1 if medoid_indices is not None else 0)
    # ncol = min(4, total_legend_items)  # You can adjust the "4" as your preferred max per row
    ax.set_xlabel("TSNE Component 1", fontdict=x_y_label_format)
    ax.set_ylabel("TSNE Component 2", fontdict=x_y_label_format)
    # ax.set_title(plot_title, fontdict=title_label_format)
    ax.tick_params(axis='both', labelsize=16)
    ax.legend(loc=loc,
              bbox_to_anchor=(0.5, 1.2),
              title=plot_title,
              title_fontsize=16,
              fontsize=14,
              ncol=ncol,
              )

    plt.tight_layout()
    plt.show()

    return tsne_results

In [7]:
def _save_results(
    target: str,
    method: str,
    metric: Union[str, Callable],
    vector: str,
    unique_clusters: np.ndarray,
    cluster_sizes: dict,
    eps: float = None,
    min_sample: int = None,
    min_cluster_size: int = None,
    cluster_selection_epsilon: float = None,
    si_score: float = None,
    db_score: float = None,
    ch_score: float = None,
) -> pd.DataFrame:
    """
    Save clustering results to a DataFrame.

    Args:
        target: Target identifier.
        method: Clustering method ('KMedoids', 'Butina', or 'ScanCluster').
        metric: Metric used for clustering (str or callable).
        vector: Vector type ('binary' or 'weighted').
        unique_clusters: Unique cluster labels.
        cluster_sizes: Dictionary of cluster sizes.
        eps: Distance threshold (for Butina and ScanCluster).
        min_sample: Minimum samples for DBSCAN (ScanCluster).
        min_cluster_size: Minimum cluster size for HDBSCAN (ScanCluster).
        cluster_selection_epsilon: Cluster selection epsilon for HDBSCAN (ScanCluster).
        si_score: Silhouette score.
        db_score: Davies-Bouldin score.
        ch_score: Calinski-Harabasz score.

    Returns:
        DataFrame containing the clustering results.
    """
    # Handle clustering-specific parameters
    if method == 'Butina':
        threshold_value = {'eps': eps}
    elif method in ['DBSCAN', 'HDBSCAN']:
        threshold_value = {
            'eps': eps,
            'min sample/size': min_sample or min_cluster_size,
            'cluster selection epsilon': cluster_selection_epsilon
        }
        # Remove None values from threshold dictionary
        threshold_value = {k: v for k, v in threshold_value.items() if v is not None}
    else:
        threshold_value = None  # KMedoids has no threshold value

    scores = pd.DataFrame([{
        'target': target,
        'clustering method': method,
        'metric': metric,
        'vector': vector,
        'number of clusters': len(unique_clusters),
        'cluster sizes': cluster_sizes,
        'threshold': threshold_value,
        'Silhouette score': si_score,
        'Davies-Bouldin score': db_score,
        'Calinski-Harabasz score': ch_score
    }])

    return scores

    

In [8]:
def get_clusters(
    fp: str,
    method: str = 'KMedoids',
    n_clusters: int = 2,
    eps: float = None,
    min_samples: int = None,
    min_cluster_size: int = None,
    cluster_selection_epsilon: float = None,
    # metric: Union[str, Callable] = None,
    plotting:bool=False,
    get_label:bool=False,
    n_comp: int = 2,  # Number of components for LDA,
    perplexity:int=20,
    loc:str='best'
) -> pd.DataFrame:
    """
    Perform clustering using the specified method (K-Medoids, Butina, DBSCAN, or HDBSCAN).

    Args:
        fp: Fingerprint identifier (e.g., 'target_vector').
        score_reservoir: DataFrame to store clustering results.
        method: Clustering method ('KMedoids', 'Butina', 'DBSCAN', or 'HDBSCAN').
        n_clusters: Number of clusters for K-Medoids.
        eps: Distance threshold for DBSCAN or HDBSCAN.
        min_samples: Minimum samples for DBSCAN or HDBSCAN.
        min_cluster_size: Minimum cluster size for HDBSCAN.
        cluster_selection_epsilon: Cluster selection epsilon for HDBSCAN.
        metric: Metric to use for clustering (str or callable).

    Returns:
        Updated score_reservoir DataFrame.
    """
    target = fp.split('_')[0]
    vector = fp.split('_')[1]
    if vector == 'ECFP'or vector=='MACCS':
      vector = '_'.join([vector,fp.split('_')[2]])
    # Determine metric based on vector type if not provided
    # if metric is None:
    if 'binary' in vector:
        metric = 'jaccard'
    elif 'count' in vector:
        metric = 'weighted_jaccard'
    else:
        metric = 'euclidean'

    print(metric)
    # Get fingerprint vector
    fp_vector = naming[fp]
    medoid_indices = None
    # Perform clustering based on the method
    if method == 'KMedoids':
        # K-Medoids clustering
        KM = KMedoids(n_clusters=n_clusters, metric=naming[metric], random_state=42, method='pam', init='k-medoids++')
        predicted_clusters = KM.fit_predict(fp_vector)
        medoid_indices = KM.medoid_indices_
    elif method == 'GaussianMixture':
          gm = GaussianMixture(n_components=n_clusters, random_state=42,max_iter=200)
          predicted_clusters = gm.fit_predict(fp_vector)
    elif method == 'Butina':

        if vector == 'binary':
          metric = 'binary_jaccard'
        elif vector == 'count':
          metric ='weighted_jaccard'

        else:
          metric ='manual_euclidean'

        # Butina clustering
        predicted_clusters = Butina.ClusterData(
            data=fp_vector,
            nPts=len(fp_vector),
            distThresh=eps,
            isDistData=False,
            distFunc=naming[metric]
        )
        # Flatten cluster labels
        cluster_labels = np.zeros(len(fp_vector), dtype=int)
        for label, cluster in enumerate(predicted_clusters):
            for index in cluster:
                cluster_labels[index] = label
        predicted_clusters = cluster_labels
    elif method in ['DBSCAN', 'HDBSCAN']:
        # DBSCAN or HDBSCAN clustering
        clustering_method = naming[method]
        parameters = {
            "metric": naming[metric],
            "eps": eps,
            "min_samples": min_samples,
            "min_cluster_size": min_cluster_size,
            "cluster_selection_epsilon": cluster_selection_epsilon
        }
        parameters = {k: v for k, v in parameters.items() if v is not None}
        dbscan = clustering_method(**parameters)
        predicted_clusters = dbscan.fit_predict(fp_vector)
    # elif method == 'LDA':
    #     # LDA clustering
    #     lda = LatentDirichletAllocation(n_components=n_comp, random_state=42)
    #     clusters = lda.fit_transform(fp_vector)
    #     predicted_clusters = np.argmax(clusters, axis=1)
    else:
        raise ValueError("Invalid method. Choose 'KMedoids', 'Butina', 'DBSCAN', 'HDBSCAN', or 'LDA'.")
    # print(len(predicted_clusters))
    # Calculate clustering scores
    si_score, db_score, ch_score = _get_scores(fp_vector, predicted_clusters, naming[metric])
    print(f"Silhouette score: {round(si_score, 2)}\nDavies-Bouldin score: {round(db_score, 2)}\nCalinski-Harabasz score: {round(ch_score, 2)}")

    # Plot clusters

    # Save results to score_reservoir
    unique_clusters, label_counts = np.unique(predicted_clusters, return_counts=True)
    cluster_sizes = {f'C_{int(label)}': int(count) for label, count in zip(unique_clusters, label_counts)}
    if plotting:
      tsne_results = plot_clusters(fp_vector,
                                   predicted_clusters,
                                   unique_clusters,
                                   method,
                                   target,
                                   vector,
                                   metric,
                                   medoid_indices,
                                   perplexity=perplexity,
                                   loc=loc
                                   )

    # Handle threshold parameters for DBSCAN and HDBSCAN
    scores = _save_results(
        target=target,
        method=method,
        metric=metric,
        vector=vector,
        unique_clusters=unique_clusters,
        cluster_sizes=cluster_sizes,
        eps=eps,
        min_sample=min_samples,
        min_cluster_size=min_cluster_size,
        cluster_selection_epsilon=cluster_selection_epsilon,
        si_score=si_score,
        db_score=db_score,
        ch_score=ch_score
    )

    if get_label:
      if method=='KMedoids':
        return scores, predicted_clusters, medoid_indices, tsne_results
      return scores, predicted_clusters, tsne_results
    return scores

In [9]:
def save_all_clustering_scores(target_vector_list, eps_range=np.arange(0.01, 1, 0.04)) -> pd.DataFrame:
    clustering_score = pd.DataFrame(columns=[
        'target',
        'metric',
        'clustering method',
        'vector',
        'number of clusters',
        'cluster sizes',
        'threshold',
        'Silhouette score',
        'Davies-Bouldin score',
        'Calinski-Harabasz score'
    ])

    for fp_vector in target_vector_list:
        # Butina
        for eps in eps_range:
            try:
                bt_score = get_clusters(fp=fp_vector, method='Butina', eps=eps)
                clustering_score = pd.concat([clustering_score, bt_score], ignore_index=True)
            except ValueError as e:
                if "Number of labels is" in str(e):
                    print(f"Skipping Butina | {fp_vector} | eps={eps} due to insufficient clusters.")
                    continue
                else:
                    raise

        # KMedoids and Gaussian Mixture
        for n_c in [2, 3, 4, 5, 6]:
            try:
                KM_score = get_clusters(fp=fp_vector, method='KMedoids', n_clusters=n_c)
                clustering_score = pd.concat([clustering_score, KM_score], ignore_index=True)
            except ValueError as e:
                if "Number of labels is" in str(e):
                    print(f"Skipping KMedoids | {fp_vector} | n_clusters={n_c} due to insufficient clusters.")
                    continue
                else:
                    raise
        if 'binary' not in fp_vector and 'count' not in fp_vector:
          print(fp_vector)
          for n_c in [2, 3, 4, 5, 6]:
              try:
                  gm_score = get_clusters(fp=fp_vector, method='GaussianMixture', n_clusters=n_c)
                  clustering_score = pd.concat([clustering_score, gm_score], ignore_index=True)
              except ValueError as e:
                  if "Number of labels is" in str(e):
                      print(f"Skipping GMM | {fp_vector} | n_clusters={n_c} due to insufficient clusters.")
                      continue
                  else:
                      raise

        # LDA
        # for n_c in [3, 4, 5, 8, 16, 32, 128]:
        #     try:
        #         L_score = get_clusters(fp=fp_vector, method='LDA', n_comp=n_c)
        #         clustering_score = pd.concat([clustering_score, L_score], ignore_index=True)
        #     except ValueError as e:
        #         if "Number of labels is" in str(e) or "Negative values in data passed" in str(e):
        #             print(f"Skipping LDA | {fp_vector} | n_comp={n_c} due to error: {str(e)}")
        #             continue
        #         else:
        #             raise

        # DBSCAN and HDBSCAN
        for given_eps in eps_range:
            for n_sample in [2, 3, 4, 5, 6]:
                try:
                    DB_score = get_clusters(fp=fp_vector, method='DBSCAN', eps=given_eps, min_samples=n_sample)
                    clustering_score = pd.concat([clustering_score, DB_score], ignore_index=True)
                except ValueError as e:
                    if "Number of labels is" in str(e):
                        print(f"Skipping DBSCAN | {fp_vector} | eps={given_eps} | min_samples={n_sample} due to insufficient clusters.")
                        continue
                    else:
                        raise

                try:
                    HDB_score = get_clusters(fp=fp_vector, method='HDBSCAN',
                                             min_cluster_size=n_sample,
                                             cluster_selection_epsilon=given_eps)
                    clustering_score = pd.concat([clustering_score, HDB_score], ignore_index=True)
                except ValueError as e:
                    if "Number of labels is" in str(e):
                        print(f"Skipping HDBSCAN | {fp_vector} | eps={given_eps} | min_samples={n_sample} due to insufficient clusters.")
                        continue
                    else:
                        raise

    return clustering_score


In [11]:
def feature_space_distance_distribution(fp,metric):

    dist_mtrx = pdist(naming[fp], metric=metric)
    plt.figure(figsize=(8, 6))
    sns.histplot(dist_mtrx, bins=30, kde=True,color="#4584b6")
    plt.title(f"{fp.split('_')[1]} Feature Distances")
    plt.xlabel(f"Pairwise {metric} Distance")
    plt.ylabel("Count")
    plt.tick_params(axis='x', )
    plt.tick_params(axis='y', )

    plt.tight_layout()
    plt.show()


# Import Data

## *Beyond molecular structure: critically assessing machine learning for designing organic photovoltaic materials and devices*

Specification

*   558 datapoints
*   143 unique donor polymers, and 261 unique acceptor molecules



https://doi.org/10.1039/D4TA01942C


In [ ]:
PCE_data = pd.read_pickle()

## Miniaturization of Popular Reactions from the Medicinal Chemists Toolbox for Ultrahigh-Throughput Experimentation

Specification

* ~768 datapoints

https://doi.org/10.1038/s44160-023-00351-1

https://github.com/cernaklab/medchem-reaction-miniaturization/blob/main/suzuki-ez.xlsx

## *Machine Learning for Polymer Design to Enhance Pervaporation-Based Organic Recovery*

**Specification**


*   52 unique polymers and 32 types of organic solutes
*   2341 datapoints

**Targets**
* total flux J (kg m–2 h–1)
* separation factor βB/A.



https://doi.org/10.1021/acs.est.4c00060

## Machine Learning-Enabled Prediction and High-Throughput Screening of Polymer Membranes for Pervaporation Separation

Specification


*   681 datapoints
*   16 unique polymers




https://doi.org/10.1021/acsami.1c22886

## Understanding and Designing a High-Performance Ultrafiltration Membrane Using Machine Learning

specification


*   320 datapoints
*   11 unique polymers



https://doi.org/10.1021/acs.est.2c05404